In [0]:
import boto3

access_key = dbutils.secrets.get(scope="aws", key="access_key")
secret_key = dbutils.secrets.get(scope="aws", key="secret_key")

s3 = boto3.client(
    "s3",
    aws_access_key_id=access_key,
    aws_secret_access_key=secret_key,
    region_name="eu-north-1"
)

bucket_name = "ysa-ecommerce-lakehouse-2026-967228660879-eu-north-1-an"

response = s3.list_objects_v2(Bucket=bucket_name, Prefix="raw/Olist/")
for obj in response.get("Contents", []):
    print(obj["Key"])

In [0]:
response = s3.list_objects_v2(Bucket=bucket_name)
for obj in response.get("Contents", []):
    print(obj["Key"])

In [0]:
import os

local_volume_path = "/Volumes/workspace/default/raw_data/olist"
os.makedirs(local_volume_path, exist_ok=True)

for obj in response.get("Contents", []):
    key = obj["Key"]
    filename = key.split("/")[-1]
    if filename:  # skip the folder placeholder itself
        s3.download_file(bucket_name, key, f"{local_volume_path}/{filename}")
        print(f"Downloaded {filename}")

In [0]:
import os
import shutil
import logging
from pathlib import Path
from typing import List, Dict, Any

import boto3
from botocore.client import BaseClient


# --- config ---
AWS_REGION = "eu-north-1"
BUCKET_NAME = "ysa-ecommerce-lakehouse-2026-967228660879-eu-north-1-an"
SOURCE_PREFIX = "raw/Olist/"

STAGING_DIR = Path("/tmp/olist_staging")
VOLUME_DIR = Path("/Volumes/workspace/default/raw_data/olist")


logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("olist_ingestion")


def get_s3_client(region: str) -> BaseClient:
    # pulling creds from secrets instead of hardcoding, obviously
    access_key = dbutils.secrets.get(scope="aws", key="access_key")
    secret_key = dbutils.secrets.get(scope="aws", key="secret_key")

    return boto3.client(
        "s3",
        aws_access_key_id=access_key,
        aws_secret_access_key=secret_key,
        region_name=region,
    )


def list_source_objects(s3_client: BaseClient, bucket: str, prefix: str) -> List[Dict[str, Any]]:
    # S3 always throws in the folder itself as a zero-byte object, filter that out
    response = s3_client.list_objects_v2(Bucket=bucket, Prefix=prefix)
    objects = response.get("Contents", [])
    files = [obj for obj in objects if not obj["Key"].endswith("/")]
    logger.info("Found %d files under s3://%s/%s", len(files), bucket, prefix)
    return files


def download_to_volume(
    s3_client: BaseClient,
    bucket: str,
    s3_key: str,
    staging_dir: Path,
    volume_dir: Path,
) -> Path:
    # can't download straight into a UC Volume - boto3 does multipart writes
    # under the hood and Volumes don't support that, throws an I/O error.
    # so we land it in /tmp first then just copy the finished file over.
    filename = s3_key.split("/")[-1]
    staging_path = staging_dir / filename
    final_path = volume_dir / filename

    if final_path.exists():
        logger.info("Already have %s, skipping", filename)
        return final_path

    s3_client.download_file(bucket, s3_key, str(staging_path))
    shutil.copy(staging_path, final_path)
    logger.info("Landed %s -> %s", filename, final_path)
    return final_path


def ingest_olist_dataset() -> List[Path]:
    staging_dir = STAGING_DIR
    volume_dir = VOLUME_DIR
    staging_dir.mkdir(parents=True, exist_ok=True)
    volume_dir.mkdir(parents=True, exist_ok=True)

    s3_client = get_s3_client(AWS_REGION)
    source_files = list_source_objects(s3_client, BUCKET_NAME, SOURCE_PREFIX)

    if not source_files:
        # usually means a typo in the prefix - S3 keys are case sensitive
        logger.warning("Nothing found under '%s', double check bucket/prefix", SOURCE_PREFIX)
        return []

    landed_paths = [
        download_to_volume(s3_client, BUCKET_NAME, obj["Key"], staging_dir, volume_dir)
        for obj in source_files
    ]
    logger.info("Done - %d files in %s", len(landed_paths), volume_dir)
    return landed_paths


ingested_files = ingest_olist_dataset()

In [0]:
display(dbutils.fs.ls(str(VOLUME_DIR)))